# Yeu cau 3: Model Versioning & Registry
Dang ky model vao MLflow Registry, gan stage, so sanh 2 version.

In [1]:
import mlflow
from mlflow.tracking import MlflowClient

client = MlflowClient()

# Lay tat ca runs trong experiment 'spam_classification'
experiment = mlflow.get_experiment_by_name('spam_classification')
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
print(runs[['tags.mlflow.runName', 'metrics.test_f1', 'metrics.test_precision', 'metrics.test_recall']])

c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  tags.mlflow.runName  metrics.test_f1  metrics.test_precision  \
0             xgboost           0.7805                  0.9697   
1         naive_bayes           0.9158                  0.9457   

   metrics.test_recall  
0               0.6531  
1               0.8878  


In [2]:
# Tim run_id cua tung model
nb_run  = runs[runs['tags.mlflow.runName'] == 'naive_bayes'].iloc[0]
xgb_run = runs[runs['tags.mlflow.runName'] == 'xgboost'].iloc[0]

# Dang ky ca hai model vao Registry
MODEL_NAME = 'SpamClassifier'

nb_uri  = f"runs:/{nb_run['run_id']}/naive_bayes_model"
xgb_uri = f"runs:/{xgb_run['run_id']}/xgboost_model"

nb_mv  = mlflow.register_model(nb_uri,  MODEL_NAME)
xgb_mv = mlflow.register_model(xgb_uri, MODEL_NAME)

print(f'Naive Bayes  -> version {nb_mv.version}')
print(f'XGBoost      -> version {xgb_mv.version}')

Successfully registered model 'SpamClassifier'.
2026/05/04 10:42:01 WARNING mlflow.tracking._model_registry.fluent: Run with id 6143beb8c67449e6a495d38d9bed440c has no artifacts at artifact path 'naive_bayes_model', registering model based on models:/m-7a728bad919d41fcaf01d29a86393cc4 instead
Created version '1' of model 'SpamClassifier'.
Registered model 'SpamClassifier' already exists. Creating a new version of this model...
2026/05/04 10:42:01 WARNING mlflow.tracking._model_registry.fluent: Run with id 07d72ee6ed82446fb292f6943a517bcb has no artifacts at artifact path 'xgboost_model', registering model based on models:/m-58eefc80fce24c3ba3b88086aaf466db instead


Naive Bayes  -> version 1
XGBoost      -> version 2


Created version '2' of model 'SpamClassifier'.


In [3]:
import time
time.sleep(3)  # Cho MLflow dang ky xong

# So sanh F1 de xac dinh model tot hon
nb_f1  = nb_run['metrics.test_f1']
xgb_f1 = xgb_run['metrics.test_f1']

print(f'Naive Bayes F1  = {nb_f1:.4f}')
print(f'XGBoost F1      = {xgb_f1:.4f}')

if xgb_f1 >= nb_f1:
    best_version    = xgb_mv.version
    staging_version = nb_mv.version
    print(f'XGBoost tot hon -> Production (v{best_version}), Naive Bayes -> Staging (v{staging_version})')
else:
    best_version    = nb_mv.version
    staging_version = xgb_mv.version
    print(f'Naive Bayes tot hon -> Production (v{best_version}), XGBoost -> Staging (v{staging_version})')

Naive Bayes F1  = 0.9158
XGBoost F1      = 0.7805
Naive Bayes tot hon -> Production (v1), XGBoost -> Staging (v2)


In [4]:
# Gan stage cho tung version
client.transition_model_version_stage(MODEL_NAME, version=best_version,    stage='Production')
client.transition_model_version_stage(MODEL_NAME, version=staging_version, stage='Staging')

print('Da gan stage xong!')
for mv in client.search_model_versions(f"name='{MODEL_NAME}'"):
    print(f'  Version {mv.version}: {mv.current_stage}')

Da gan stage xong!
  Version 2: Staging
  Version 1: Production


C:\Users\Admin\AppData\Local\Temp\ipykernel_15484\1658160959.py:2: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(MODEL_NAME, version=best_version,    stage='Production')
C:\Users\Admin\AppData\Local\Temp\ipykernel_15484\1658160959.py:3: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(MODEL_NAME, version=staging_version, stage=

## Giai thich ngan (3-5 dong)

Ca hai model duoc dang ky vao MLflow Model Registry voi ten 'SpamClassifier'.
Model co F1-score cao hon tren tap test duoc gan stage **Production**,
model con lai duoc gan **Staging** de tiep tuc theo doi.
XGBoost thuong dat F1 cao hon Naive Bayes vi no su dung ensemble gradient boosting,
ket hop nhieu cay quyet dinh de giam sai so du doan.
Naive Bayes don gian hon nhung van hieu qua voi van ban ngan va du lieu khong can bang.